# Ant Colony Optimization: Pheromone Trails

Ant Colony Optimization, or ACO, is a swarm-intelligence algorithm. It solves graph traversal problems by simulating how simple ants can collectively discover strong routes.

Marco Dorigo and collaborators developed ACO in the 1990s after studying how real ant colonies reinforce useful paths with pheromones. The algorithm has been used for routing, scheduling, vehicle paths, network design, and other hard optimization problems.

Each ant is simple:

- wander through a graph
- prefer short edges and strong pheromone trails
- leave pheromone on successful paths
- let old pheromone evaporate

Over time, a high-pheromone highway can emerge from noisy exploration.

<details>
<summary>Big idea</summary>

Shorter successful paths get completed more often, so they receive pheromone sooner and more repeatedly. That feedback loop attracts more ants.

</details>

## 1. The Mental Model

We will build a tiny map with a **nest**, a **food** node, and obstacle-like detours between them.

ACO uses two signals when an ant chooses the next edge:

- **pheromone**: what previous successful ants reinforced
- **visibility**: how attractive a short edge is, often `1 / distance`

A common scoring rule is:

```text
score(edge) = pheromone^alpha * visibility^beta
```

<details>
<summary>Parameter hint</summary>

`alpha` controls how much ants trust pheromone. `beta` controls how much ants prefer short edges. Evaporation keeps old trails from dominating forever.

</details>

## 2. Build the Objects

Implementation plan:

1. `TrailEdge` stores one undirected graph edge.
2. `AntPath` records one ant's route from nest to food.
3. `ColonyStep` stores one iteration snapshot.
4. `AntColonyMap` owns distances and pheromone levels.
5. `AntColonyRunner` sends out ants, evaporates pheromone, and deposits new pheromone.
6. `ColonyReplay` prints the trail evolution.

<details>
<summary>Implementation hint</summary>

The ants are random, but not equally random. They use weighted choices, where shorter and more pheromone-rich edges are more likely.

</details>

**Object model.** Define `TrailEdge`, the named objects used by the next examples.


In [ ]:
from dataclasses import dataclass

import random

@dataclass(frozen=True)
class TrailEdge:
    left: str
    right: str
    distance: float

    def key(self) -> tuple[str, str]:
        return tuple(sorted((self.left, self.right)))

    def other(self, node: str) -> str:
        if node == self.left:
            return self.right
        if node == self.right:
            return self.left
        raise ValueError(f"{node} is not on edge {self.left}-{self.right}.")

    def __str__(self) -> str:
        return f"{self.left}-{self.right} ({self.distance:g})"


**Trace model.** Define `AntPath`, `ColonyStep`, the structure used to capture replayable algorithm state.


In [ ]:
@dataclass(frozen=True)
class AntPath:
    nodes: tuple[str, ...]
    edge_keys: tuple[tuple[str, str], ...]
    length: float
    reached_food: bool

    def route(self) -> str:
        return " -> ".join(self.nodes)

@dataclass(frozen=True)
class ColonyStep:
    iteration: int
    successful_paths: tuple[AntPath, ...]
    best_path: AntPath | None
    pheromones: dict[tuple[str, str], float]
    note: str


**Object model.** Define `AntColonyMap`, the named objects used by the next examples.


In [ ]:
class AntColonyMap:
    def __init__(self, edges: list[TrailEdge], start: str, goal: str, initial_pheromone: float = 1.0):
        self.start = start
        self.goal = goal
        self.edges = {edge.key(): edge for edge in edges}
        self.pheromones = {edge.key(): initial_pheromone for edge in edges}
        self.neighbors: dict[str, list[TrailEdge]] = {}

        for edge in edges:
            self.neighbors.setdefault(edge.left, []).append(edge)
            self.neighbors.setdefault(edge.right, []).append(edge)

    def edge(self, key: tuple[str, str]) -> TrailEdge:
        return self.edges[tuple(sorted(key))]

    def pheromone(self, edge: TrailEdge) -> float:
        return self.pheromones[edge.key()]

    def snapshot(self) -> dict[tuple[str, str], float]:
        return {key: round(value, 3) for key, value in self.pheromones.items()}

    def reset_pheromones(self, value: float = 1.0) -> None:
        for key in self.pheromones:
            self.pheromones[key] = value


**Algorithm engine.** Define `AntColonyRunner`, the class that runs the main simulation or algorithm.


In [ ]:
class AntColonyRunner:
    def __init__(
        self,
        colony_map: AntColonyMap,
        ant_count: int = 18,
        alpha: float = 1.0,
        beta: float = 2.0,
        evaporation_rate: float = 0.25,
        deposit_strength: float = 18.0,
        seed: int = 7,
    ):
        self.colony_map = colony_map
        self.ant_count = ant_count
        self.alpha = alpha
        self.beta = beta
        self.evaporation_rate = evaporation_rate
        self.deposit_strength = deposit_strength
        self.rng = random.Random(seed)

    def _weighted_choice(self, choices: list[tuple[TrailEdge, float]]) -> TrailEdge:
        total = sum(weight for _, weight in choices)
        pick = self.rng.uniform(0, total)
        running = 0.0

        for edge, weight in choices:
            running += weight
            if running >= pick:
                return edge

        return choices[-1][0]

    def _walk_ant(self, max_steps: int) -> AntPath:
        current = self.colony_map.start
        nodes = [current]
        edge_keys: list[tuple[str, str]] = []
        visited = {current}
        length = 0.0

        for _ in range(max_steps):
            if current == self.colony_map.goal:
                break

            choices = []
            for edge in self.colony_map.neighbors.get(current, []):
                next_node = edge.other(current)
                if next_node in visited and next_node != self.colony_map.goal:
                    continue

                pheromone = self.colony_map.pheromone(edge) ** self.alpha
                visibility = (1 / edge.distance) ** self.beta
                choices.append((edge, pheromone * visibility))

            if not choices:
                break

            chosen_edge = self._weighted_choice(choices)
            current = chosen_edge.other(current)
            nodes.append(current)
            edge_keys.append(chosen_edge.key())
            visited.add(current)
            length += chosen_edge.distance

        return AntPath(
            nodes=tuple(nodes),
            edge_keys=tuple(edge_keys),
            length=length,
            reached_food=current == self.colony_map.goal,
        )

    def _evaporate(self) -> None:
        keep_rate = 1 - self.evaporation_rate
        for key in self.colony_map.pheromones:
            self.colony_map.pheromones[key] *= keep_rate

    def _deposit(self, paths: list[AntPath]) -> None:
        for path in paths:
            if not path.reached_food or path.length == 0:
                continue
            deposit = self.deposit_strength / path.length
            for key in path.edge_keys:
                self.colony_map.pheromones[key] += deposit

    def run(self, iterations: int = 12) -> list[ColonyStep]:
        steps: list[ColonyStep] = []
        best_path: AntPath | None = None
        max_steps = len(self.colony_map.neighbors) + 2

        for iteration in range(1, iterations + 1):
            paths = [self._walk_ant(max_steps=max_steps) for _ in range(self.ant_count)]
            successful_paths = [path for path in paths if path.reached_food]

            for path in successful_paths:
                if best_path is None or path.length < best_path.length:
                    best_path = path

            self._evaporate()
            self._deposit(successful_paths)

            steps.append(
                ColonyStep(
                    iteration=iteration,
                    successful_paths=tuple(successful_paths),
                    best_path=best_path,
                    pheromones=self.colony_map.snapshot(),
                    note=f"{len(successful_paths)} of {self.ant_count} ants reached food.",
                )
            )

        return steps


## 3. Build the Nest-to-Food Graph

The graph has several possible routes. Some are short, some are longer obstacle detours.

The best route should be:

```text
Nest -> Fern -> Bridge -> Food
```

<details>
<summary>Obstacle model</summary>

We do not need walls on a grid. Longer graph edges act like rough terrain, stones, mud, or slow crossings that make a path less attractive.

</details>

In [2]:
trail_edges = [
    TrailEdge("Nest", "Fern", 2),
    TrailEdge("Fern", "Bridge", 2),
    TrailEdge("Bridge", "Food", 3),
    TrailEdge("Nest", "Pebble", 4),
    TrailEdge("Pebble", "Bridge", 3),
    TrailEdge("Pebble", "Food", 8),
    TrailEdge("Nest", "Mud", 3),
    TrailEdge("Mud", "Tunnel", 4),
    TrailEdge("Tunnel", "Food", 4),
    TrailEdge("Mud", "Bridge", 5),
    TrailEdge("Nest", "Flower", 7),
    TrailEdge("Flower", "Food", 2),
]

colony_map = AntColonyMap(trail_edges, start="Nest", goal="Food", initial_pheromone=1.0)

print("Trail map:")
for edge in trail_edges:
    print(f"  {edge}")

Trail map:
  Nest-Fern (2)
  Fern-Bridge (2)
  Bridge-Food (3)
  Nest-Pebble (4)
  Pebble-Bridge (3)
  Pebble-Food (8)
  Nest-Mud (3)
  Mud-Tunnel (4)
  Tunnel-Food (4)
  Mud-Bridge (5)
  Nest-Flower (7)
  Flower-Food (2)


## 4. Run the Colony

Each iteration sends out several ants. Successful ants deposit pheromone based on path quality, while every edge loses some pheromone through evaporation.

<details>
<summary>Update rule</summary>

After each iteration, every edge is multiplied by `1 - evaporation_rate`. Then each successful path adds `deposit_strength / path_length` to every edge it used.

</details>

In [3]:
runner = AntColonyRunner(
    colony_map=colony_map,
    ant_count=24,
    alpha=1.2,
    beta=2.0,
    evaporation_rate=0.28,
    deposit_strength=20.0,
    seed=12,
)
steps = runner.run(iterations=14)
final_step = steps[-1]

print(f"Final iteration: {final_step.note}")
print(f"Best route: {final_step.best_path.route()}")
print(f"Best length: {final_step.best_path.length:g}")

print("\nStrongest pheromone edges:")
for key, amount in sorted(final_step.pheromones.items(), key=lambda item: item[1], reverse=True)[:5]:
    edge = colony_map.edge(key)
    print(f"  {edge.left}-{edge.right}: {amount:.2f}")

Final iteration: 24 of 24 ants reached food.
Best route: Nest -> Fern -> Bridge -> Food
Best length: 7

Strongest pheromone edges:
  Nest-Fern: 240.71
  Fern-Bridge: 240.71
  Bridge-Food: 239.61
  Pebble-Bridge: 1.07
  Pebble-Food: 1.07


## 5. Replay the Pheromone Highway

A replay shows the feedback loop: early randomness, repeated successful routes, then strong pheromone concentration on the best path.

<details>
<summary>Reading the replay</summary>

If the same few edges keep appearing at the top, the colony is converging. If many edges stay similar, the ants are still exploring.

</details>

In [4]:
class ColonyReplay:
    def __init__(self, colony_map: AntColonyMap, steps: list[ColonyStep]):
        self.colony_map = colony_map
        self.steps = steps

    def show(self, selected_iterations: list[int]) -> None:
        for iteration in selected_iterations:
            step = self.steps[iteration - 1]
            best = step.best_path.route() if step.best_path else "none yet"
            print(f"Iteration {step.iteration}: {step.note}")
            print(f"  best route: {best}")
            print("  strongest trails:")
            for key, amount in sorted(step.pheromones.items(), key=lambda item: item[1], reverse=True)[:4]:
                edge = self.colony_map.edge(key)
                print(f"    {edge.left}-{edge.right}: {amount:.2f}")
            print()


ColonyReplay(colony_map, steps).show([1, 3, 7, 14])

Iteration 1: 22 of 24 ants reached food.
  best route: Nest -> Fern -> Bridge -> Food
  strongest trails:
    Nest-Fern: 38.31
    Fern-Bridge: 38.31
    Bridge-Food: 31.11
    Mud-Tunnel: 8.53

Iteration 3: 24 of 24 ants reached food.
  best route: Nest -> Fern -> Bridge -> Food
  strongest trails:
    Nest-Fern: 123.66
    Fern-Bridge: 123.66
    Bridge-Food: 113.16
    Pebble-Bridge: 9.75

Iteration 7: 24 of 24 ants reached food.
  best route: Nest -> Fern -> Bridge -> Food
  strongest trails:
    Nest-Fern: 209.29
    Fern-Bridge: 209.29
    Bridge-Food: 203.82
    Pebble-Bridge: 5.27

Iteration 14: 24 of 24 ants reached food.
  best route: Nest -> Fern -> Bridge -> Food
  strongest trails:
    Nest-Fern: 240.71
    Fern-Bridge: 240.71
    Bridge-Food: 239.61
    Pebble-Bridge: 1.07



## 6. NetworkX Pheromone Visualizer

Now we draw the graph. Stronger pheromone trails become thicker and brighter, so the best path looks like an emerging highway.

<details>
<summary>Visualizer hint</summary>

The layout is fixed by hand so the graph does not jump around between iterations. Only the edge color and width change.

</details>

In [5]:
visualizer_ready = False

try:
    import matplotlib.pyplot as plt
    import networkx as nx
    visualizer_ready = True
    print("NetworkX and Matplotlib are ready.")
except ModuleNotFoundError as error:
    print(f"Missing package: {error.name}")
    print("Install inside a notebook with: %pip install networkx matplotlib")

NetworkX and Matplotlib are ready.


**Builder helper.** Define `build_networkx_graph`, which prepares reusable examples or traces.


In [ ]:
node_positions = {
    "Nest": (0, 0),
    "Fern": (1, 1),
    "Bridge": (2.4, 1),
    "Food": (3.8, 0),
    "Pebble": (1.3, -0.3),
    "Mud": (0.9, -1.2),
    "Tunnel": (2.4, -1.4),
    "Flower": (1.8, 1.8),
}

def build_networkx_graph(edges: list[TrailEdge]):
    graph = nx.Graph()
    for edge in edges:
        graph.add_edge(edge.left, edge.right, distance=edge.distance, key=edge.key())
    return graph


**Visual helper.** Define `draw_pheromone_snapshot`, which turns state into something students can inspect.


In [ ]:
def draw_pheromone_snapshot(graph, step: ColonyStep, title: str, ax) -> None:
    edge_order = list(graph.edges())
    pheromone_values = [step.pheromones[tuple(sorted(edge))] for edge in edge_order]
    max_pheromone = max(pheromone_values) if pheromone_values else 1
    widths = [1 + 7 * (value / max_pheromone) for value in pheromone_values]

    ax.set_title(title)
    ax.axis("off")

    nx.draw_networkx_nodes(
        graph,
        node_positions,
        node_color=["#f5d76e" if node in ["Nest", "Food"] else "#d9f0d3" for node in graph.nodes],
        edgecolors="#2f3e46",
        linewidths=1.5,
        node_size=1150,
        ax=ax,
    )
    nx.draw_networkx_labels(graph, node_positions, font_size=9, font_weight="bold", ax=ax)
    nx.draw_networkx_edges(
        graph,
        node_positions,
        edgelist=edge_order,
        edge_color=pheromone_values,
        edge_cmap=plt.cm.plasma,
        edge_vmin=0,
        edge_vmax=max_pheromone,
        width=widths,
        alpha=0.9,
        ax=ax,
    )

    edge_labels = nx.get_edge_attributes(graph, "distance")
    nx.draw_networkx_edge_labels(graph, node_positions, edge_labels=edge_labels, font_size=8, ax=ax)


**Inspect the result.** Render the current state so the algorithm is easier to reason about.


In [ ]:
graph = build_networkx_graph(trail_edges)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

draw_pheromone_snapshot(graph, steps[0], "Iteration 1: noisy exploration", axes[0])

draw_pheromone_snapshot(graph, steps[-1], "Iteration 14: pheromone highway", axes[1])

fig.suptitle("Ant Colony Optimization: thicker and brighter edges have stronger pheromone", fontsize=12)

plt.tight_layout()

plt.show()


## 7. Experiments

ACO is controlled by feedback settings. Change `alpha`, `beta`, and evaporation to see whether ants explore widely or lock onto a route quickly.

<details>
<summary>Experiment hint</summary>

Higher `beta` favors short edges more strongly. Higher `alpha` favors existing pheromone more strongly. Higher evaporation clears old trails faster.

</details>

**Simulation helper.** Define `run_colony_experiment`, which advances the model and records behavior.


In [ ]:
def run_colony_experiment(label: str, alpha: float, beta: float, evaporation_rate: float, seed: int) -> None:
    experiment_map = AntColonyMap(trail_edges, start="Nest", goal="Food", initial_pheromone=1.0)
    experiment_runner = AntColonyRunner(
        colony_map=experiment_map,
        ant_count=20,
        alpha=alpha,
        beta=beta,
        evaporation_rate=evaporation_rate,
        deposit_strength=20.0,
        seed=seed,
    )
    experiment_steps = experiment_runner.run(iterations=10)
    final = experiment_steps[-1]
    best = final.best_path.route() if final.best_path else "none"

    print(label)
    print(f"  alpha={alpha}, beta={beta}, evaporation={evaporation_rate}")
    print(f"  best route: {best}")
    print("  top trails:")
    for key, amount in sorted(final.pheromones.items(), key=lambda item: item[1], reverse=True)[:3]:
        edge = experiment_map.edge(key)
        print(f"    {edge.left}-{edge.right}: {amount:.2f}")
    print()


**Inspect the result.** Evaluate the expression and read the output before changing parameters.


In [ ]:
run_colony_experiment("Short-edge bias", alpha=1.0, beta=3.0, evaporation_rate=0.25, seed=4)

run_colony_experiment("Pheromone-heavy bias", alpha=2.0, beta=1.0, evaporation_rate=0.15, seed=4)

run_colony_experiment("Fast evaporation", alpha=1.2, beta=2.0, evaporation_rate=0.55, seed=4)


## What You Should Remember

Ant Colony Optimization is a feedback loop:

- Ants explore possible paths through a graph.
- Shorter paths are more attractive through visibility.
- Successful paths receive pheromone deposits.
- Evaporation prevents old paths from lasting forever.
- Repeated reinforcement can turn a noisy search into a strong route.

<details>
<summary>Where this shows up</summary>

ACO is used for routing, scheduling, path planning, traveling-salesperson-style problems, and other graph search problems where many simple agents can explore candidate solutions.

</details>

## Visual Trace + Rigor Studio

**Problem frame.** Use local agents and pheromone memory to search hard route spaces.

**Interactive animation target.** Animate ant tours, pheromone evaporation, and reinforced edges.

**Correctness handle.** Edges used by better solutions receive stronger reinforcement after each generation.

**Complexity handle.** Budget-driven; cost scales with ants, iterations, and path construction length.

**Failure mode to test.** Too much reinforcement too early can converge prematurely on a poor route.

**Studio task.** Change evaporation and explain whether exploration or exploitation dominates.


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware import AlgorithmPlayer, AlgorithmTrace, TraceStep, render_trace_table

# Convert the implementation above into snapshots:
# trace = AlgorithmTrace("Topic trace")
# trace.append("start", {"your_state": ...}, "What changed?", invariant="What remains true?")
# AlgorithmPlayer(trace, your_renderer).display()
print("Use AlgorithmTrace to turn this notebook's algorithm into a step-by-step visual player.")
